In [5]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 3 – Location problems
#  Section: Exercise 4
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP        # Modeling language
using HiGHS       # Solver
using CSV         # For reading CSV files
using DataFrames  # For handling DataFrame operations
using Distances   # For Jaccard distance calculations

# Include utility functions for plotting and distance matrix creation
include("utils/sclp_utils.jl")

# Main function to solve SCLP with overlap control
function solve_sclp_overlap(file_path; radius = 200, z = 1)
    # Load coordinates
    coordinates = CSV.read(file_path, header=true, DataFrame) |> Matrix{Float64}
    
    # Create distance matrix
    D = create_distance_matrix(coordinates)
    
    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Calculate jaccard's matrix between points coverages
    Ψ = Distances.pairwise(Jaccard(), eachrow(C))

    # Number of demand/facility points
    n = size(C, 1)

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[1:n], Bin)
    @variable(model, u)

    # Objective: minimize number of facilities opened
    @objective(model, Min, sum(x) + u)

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in 1:n], sum(C[i,j] * x[j] for j in 1:n) >= 1)

    # Constraint: Jaccard distance condition
    @constraint(model, sum((x[i] + x[j] - 1) * abs(Ψ[i,j] - z) for i in 1:n, j in 1:n) <= u)

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vector
    x_vals = JuMP.value.(x)

    # Get indices of selected facilities
    selected_facilities = findall(xi -> xi > 0.5, x_vals)

    # Display solution
    println("Number of facilities opened: ", length(selected_facilities))
    println("Selected facility indices: ", selected_facilities)

    # Display solution on map
    map = plot_solution(selected_facilities, coordinates, radius)
    display(map)
end

# Example usage (z = 1)
solve_sclp_overlap("data/sjc_coordinates.csv", radius = 400, z = 1)

PyObject <folium.folium.Map object at 0x0000021394689250>

Number of facilities opened: 4
Selected facility indices: [5, 43, 50, 97]


In [6]:
# Example usage (z = 0)
solve_sclp_overlap("data/sjc_coordinates.csv", radius = 400, z = 0)

PyObject <folium.folium.Map object at 0x0000021393E309B0>

Number of facilities opened: 4
Selected facility indices: [1, 7, 12, 73]
